### Installing and import everything

In [46]:
%pip install requests beautifulsoup4 pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [47]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import time
import os

print("All libraries imported successfully!")

All libraries imported successfully!


### Web Scraping

In [49]:
test_url = "https://scrapingsandbox.com/product/1"

response = requests.get(
    test_url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [50]:
soup = BeautifulSoup(response.text, "html.parser")

json_section = soup.find("pre")

if json_section is not None:
    test_product = json.loads(json_section.get_text())
    print("Product data found successfully!")
    print(test_product)
else:
    print("Product data was not found.")

Product data found successfully!
{'id': 1, 'title': 'Lightweight Probiotics', 'vendor': 'SoundWave', 'category': 'Health', 'price': 155.62, 'compareAtPrice': 206.69, 'description': 'High-quality lightweight probiotics from SoundWave. Perfect for everyday use. Features premium materials and thoughtful design. This health product has been carefully crafted to meet the highest standards of quality and performance.', 'image': 'https://placehold.co/400x400/10b981/ffffff?text=Probiotics', 'tags': ['natural', 'fitness'], 'sku': 'SKU-HEA-0001', 'inStock': True, 'rating': 4, 'reviewCount': 52, 'createdAt': '2022-08-29T00:00:00.000Z', 'variants': [{'color': 'Pink', 'size': 'XXL', 'sku': 'SKU-HEA-0001-PIN-XXL', 'inStock': True, 'price': 153.43}, {'color': 'Pink', 'size': 'XS', 'sku': 'SKU-HEA-0001-PIN-XS', 'inStock': True, 'price': 160.25}, {'color': 'White', 'size': 'L', 'sku': 'SKU-HEA-0001-WHI-L', 'inStock': False, 'price': 153.31}, {'color': 'Navy', 'size': 'XL', 'sku': 'SKU-HEA-0001-NAV-XL',

### Keeping only required columns 

In [51]:
test_record = {
    "product_id": test_product.get("id"),
    "product_name": test_product.get("title"),
    "vendor": test_product.get("vendor"),
    "category": test_product.get("category"),
    "price": test_product.get("price"),
    "original_price": test_product.get("compareAtPrice"),
    "rating": test_product.get("rating"),
    "review_count": test_product.get("reviewCount"),
    "in_stock": test_product.get("inStock"),
    "sku": test_product.get("sku"),
    "tags": ", ".join(test_product.get("tags", [])),
    "description": test_product.get("description"),
    "created_at": test_product.get("createdAt"),
    "image_url": test_product.get("image"),
    "product_url": test_url
}

In [52]:
test_df = pd.DataFrame([test_record])

test_df

,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,tags,description,created_at,image_url,product_url
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4,52,True,SKU-HEA-0001,"natural, fitness",High-quality lightweight probiotics from Sound...,2022-08-29T00:00:00.000Z,https://placehold.co/400x400/10b981/ffffff?tex...,https://scrapingsandbox.com/product/1


### Scraping all 500 products

In [53]:
base_url = "https://scrapingsandbox.com/product/"

products = []
failed_product_ids = []

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/136.0 Safari/537.36"
    )
})

for product_id in range(1, 501):

    product_url = f"{base_url}{product_id}"
    product_scraped = False

    # Try each product up to three times
    for attempt in range(1, 4):

        try:
            response = session.get(
                product_url,
                timeout=30
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            json_section = soup.find("pre")

            if json_section is None:
                raise ValueError("Product JSON was not found")

            product_data = json.loads(
                json_section.get_text()
            )

            product_record = {
                "product_id": product_data.get("id"),
                "product_name": product_data.get("title"),
                "vendor": product_data.get("vendor"),
                "category": product_data.get("category"),
                "price": product_data.get("price"),
                "original_price": product_data.get(
                    "compareAtPrice"
                ),
                "rating": product_data.get("rating"),
                "review_count": product_data.get(
                    "reviewCount"
                ),
                "in_stock": product_data.get("inStock"),
                "sku": product_data.get("sku"),
                "tags": ", ".join(
                    product_data.get("tags", [])
                ),
                "description": product_data.get(
                    "description"
                ),
                "created_at": product_data.get(
                    "createdAt"
                ),
                "image_url": product_data.get("image"),
                "product_url": product_url
            }

            products.append(product_record)
            product_scraped = True
            break

        except (
            requests.RequestException,
            ValueError,
            json.JSONDecodeError
        ) as error:

            print(
                f"Product {product_id}, "
                f"attempt {attempt} failed: {error}"
            )

            time.sleep(2)

    if not product_scraped:
        failed_product_ids.append(product_id)

    # Show progress and save a backup
    if product_id % 50 == 0:
        backup_df = pd.DataFrame(products)

        backup_df.to_csv(
            "products_backup.csv",
            index=False
        )

        print(
            f"Progress: {product_id}/500 pages checked, "
            f"{len(products)} products collected"
        )

    # Prevent excessive requests
    time.sleep(0.30)

Progress: 50/500 pages checked, 50 products collected
Progress: 100/500 pages checked, 100 products collected
Progress: 150/500 pages checked, 150 products collected
Progress: 200/500 pages checked, 200 products collected
Progress: 250/500 pages checked, 250 products collected
Progress: 300/500 pages checked, 300 products collected
Progress: 350/500 pages checked, 350 products collected
Progress: 400/500 pages checked, 400 products collected
Progress: 450/500 pages checked, 450 products collected
Progress: 500/500 pages checked, 500 products collected


### Check the scraping result

In [55]:
print("Products collected:", len(products))
print("Failed product IDs:", failed_product_ids)

Products collected: 500
Failed product IDs: []


In [59]:
df = pd.DataFrame(products)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (500, 15)


,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,tags,description,created_at,image_url,product_url
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4.0,52,True,SKU-HEA-0001,"natural, fitness",High-quality lightweight probiotics from Sound...,2022-08-29T00:00:00.000Z,https://placehold.co/400x400/10b981/ffffff?tex...,https://scrapingsandbox.com/product/1
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,4.6,12,False,SKU-ELE-0002,"tech, digital, USB",High-quality wireless led desk lamp from Sound...,2023-06-04T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/2
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,3.5,211,True,SKU-FOO-0003,organic,High-quality deluxe tea collection from StyleC...,2023-04-17T00:00:00.000Z,https://placehold.co/400x400/84cc16/ffffff?tex...,https://scrapingsandbox.com/product/3
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,3.6,47,False,SKU-AUT-0004,"car, accessories, travel",High-quality essential tire gauge from TechVau...,2022-10-24T00:00:00.000Z,https://placehold.co/400x400/6366f1/ffffff?tex...,https://scrapingsandbox.com/product/4
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,3.5,353,True,SKU-ART-0005,"drawing, painting",High-quality smart carving tools from SparkleB...,2022-08-07T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/5


### Inspecting the raw dataset

In [61]:
df.head()

,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,tags,description,created_at,image_url,product_url
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4.0,52,True,SKU-HEA-0001,"natural, fitness",High-quality lightweight probiotics from Sound...,2022-08-29T00:00:00.000Z,https://placehold.co/400x400/10b981/ffffff?tex...,https://scrapingsandbox.com/product/1
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,4.6,12,False,SKU-ELE-0002,"tech, digital, USB",High-quality wireless led desk lamp from Sound...,2023-06-04T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/2
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,3.5,211,True,SKU-FOO-0003,organic,High-quality deluxe tea collection from StyleC...,2023-04-17T00:00:00.000Z,https://placehold.co/400x400/84cc16/ffffff?tex...,https://scrapingsandbox.com/product/3
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,3.6,47,False,SKU-AUT-0004,"car, accessories, travel",High-quality essential tire gauge from TechVau...,2022-10-24T00:00:00.000Z,https://placehold.co/400x400/6366f1/ffffff?tex...,https://scrapingsandbox.com/product/4
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,3.5,353,True,SKU-ART-0005,"drawing, painting",High-quality smart carving tools from SparkleB...,2022-08-07T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/5


In [62]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 500
Columns: 15


In [63]:
df.columns.tolist()

['product_id',
 'product_name',
 'vendor',
 'category',
 'price',
 'original_price',
 'rating',
 'review_count',
 'in_stock',
 'sku',
 'tags',
 'description',
 'created_at',
 'image_url',
 'product_url']

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      500 non-null    int64  
 1   product_name    500 non-null    object 
 2   vendor          500 non-null    object 
 3   category        500 non-null    object 
 4   price           500 non-null    float64
 5   original_price  189 non-null    float64
 6   rating          500 non-null    float64
 7   review_count    500 non-null    int64  
 8   in_stock        500 non-null    bool   
 9   sku             500 non-null    object 
 10  tags            500 non-null    object 
 11  description     500 non-null    object 
 12  created_at      500 non-null    object 
 13  image_url       500 non-null    object 
 14  product_url     500 non-null    object 
dtypes: bool(1), float64(3), int64(2), object(9)
memory usage: 55.3+ KB


In [65]:
df.isnull().sum()

product_id          0
product_name        0
vendor              0
category            0
price               0
original_price    311
rating              0
review_count        0
in_stock            0
sku                 0
tags                0
description         0
created_at          0
image_url           0
product_url         0
dtype: int64

In [66]:
print("Duplicate rows:", df.duplicated().sum())
print(
    "Duplicate product IDs:",
    df["product_id"].duplicated().sum()
)
print(
    "Duplicate SKUs:",
    df["sku"].duplicated().sum()
)

Duplicate rows: 0
Duplicate product IDs: 0
Duplicate SKUs: 0


In [67]:
df[
    ["price", "original_price", "rating", "review_count"]
].describe()

,price,original_price,rating,review_count
count,500.000000,189.000000,500.000000,500.000000
mean,104.100060,161.359312,4.016200,234.272000
std,58.274374,88.917038,0.589825,141.336003
min,5.120000,7.760000,3.000000,0.000000
25%,55.305000,80.020000,3.500000,109.750000
50%,105.230000,175.370000,4.050000,228.000000
75%,152.625000,237.510000,4.600000,337.250000
max,204.860000,330.470000,5.000000,499.000000


In [68]:
df

,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,tags,description,created_at,image_url,product_url
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4.0,52,True,SKU-HEA-0001,"natural, fitness",High-quality lightweight probiotics from Sound...,2022-08-29T00:00:00.000Z,https://placehold.co/400x400/10b981/ffffff?tex...,https://scrapingsandbox.com/product/1
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,4.6,12,False,SKU-ELE-0002,"tech, digital, USB",High-quality wireless led desk lamp from Sound...,2023-06-04T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/2
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,3.5,211,True,SKU-FOO-0003,organic,High-quality deluxe tea collection from StyleC...,2023-04-17T00:00:00.000Z,https://placehold.co/400x400/84cc16/ffffff?tex...,https://scrapingsandbox.com/product/3
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,3.6,47,False,SKU-AUT-0004,"car, accessories, travel",High-quality essential tire gauge from TechVau...,2022-10-24T00:00:00.000Z,https://placehold.co/400x400/6366f1/ffffff?tex...,https://scrapingsandbox.com/product/4
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,3.5,353,True,SKU-ART-0005,"drawing, painting",High-quality smart carving tools from SparkleB...,2022-08-07T00:00:00.000Z,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Advanced Mystery Novel,CozyCorner,Books,9.72,11.95,3.1,155,True,SKU-BOO-0496,"reading, education, bestseller",High-quality advanced mystery novel from CozyC...,2022-08-17T00:00:00.000Z,https://placehold.co/400x400/ef4444/ffffff?tex...,https://scrapingsandbox.com/product/496
496,497,Vintage Cat Toy,HomeNest,Pet Supplies,99.70,NaN,4.8,337,True,SKU-PET-0497,"pets, food",High-quality vintage cat toy from HomeNest. Pe...,2022-08-15T00:00:00.000Z,https://placehold.co/400x400/ef4444/ffffff?tex...,https://scrapingsandbox.com/product/497
497,498,Compact Clock,HomeNest,Home & Garden,67.89,NaN,4.9,103,True,SKU-HOM-0498,"kitchen, living room",High-quality compact clock from HomeNest. Perf...,2022-09-12T00:00:00.000Z,https://placehold.co/400x400/84cc16/ffffff?tex...,https://scrapingsandbox.com/product/498
498,499,Heavy-Duty Gym Bag,TechVault,Sports,63.70,NaN,4.8,194,True,SKU-SPO-0499,"fitness, workout, training",High-quality heavy-duty gym bag from TechVault...,2023-12-24T00:00:00.000Z,https://placehold.co/400x400/3b82f6/ffffff?tex...,https://scrapingsandbox.com/product/499


### cleaning copy and clean text columns

In [70]:
clean_df = df.copy()

In [71]:
text_columns = [
    "product_name",
    "vendor",
    "category",
    "sku",
    "tags",
    "description",
    "image_url",
    "product_url"
]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
    )

In [72]:
clean_df[text_columns].head()

,product_name,vendor,category,sku,tags,description,image_url,product_url
0,Lightweight Probiotics,SoundWave,Health,SKU-HEA-0001,"natural, fitness",High-quality lightweight probiotics from Sound...,https://placehold.co/400x400/10b981/ffffff?tex...,https://scrapingsandbox.com/product/1
1,Wireless LED Desk Lamp,SoundWave,Electronics,SKU-ELE-0002,"tech, digital, USB",High-quality wireless led desk lamp from Sound...,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/2
2,Deluxe Tea Collection,StyleCraft,Food & Beverage,SKU-FOO-0003,organic,High-quality deluxe tea collection from StyleC...,https://placehold.co/400x400/84cc16/ffffff?tex...,https://scrapingsandbox.com/product/3
3,Essential Tire Gauge,TechVault,Automotive,SKU-AUT-0004,"car, accessories, travel",High-quality essential tire gauge from TechVau...,https://placehold.co/400x400/6366f1/ffffff?tex...,https://scrapingsandbox.com/product/4
4,Smart Carving Tools,SparkleBox,Art,SKU-ART-0005,"drawing, painting",High-quality smart carving tools from SparkleB...,https://placehold.co/400x400/f59e0b/ffffff?tex...,https://scrapingsandbox.com/product/5


In [73]:
print("Original shape:", df.shape)
print("Cleaning copy shape:", clean_df.shape)

Original shape: (500, 15)
Cleaning copy shape: (500, 15)


### Check and remove duplicates

In [74]:
print(
    "Duplicate rows:",
    clean_df.duplicated().sum()
)

print(
    "Duplicate product IDs:",
    clean_df["product_id"].duplicated().sum()
)

print(
    "Duplicate SKUs:",
    clean_df["sku"].duplicated().sum()
)

Duplicate rows: 0
Duplicate product IDs: 0
Duplicate SKUs: 0


In [75]:
clean_df = clean_df.drop_duplicates()

clean_df = clean_df.drop_duplicates(
    subset=["product_id"],
    keep="first"
)

clean_df = clean_df.drop_duplicates(
    subset=["sku"],
    keep="first"
)

clean_df = clean_df.reset_index(drop=True)

In [76]:
print("Final shape:", clean_df.shape)
print("Duplicate rows:", clean_df.duplicated().sum())
print(
    "Duplicate product IDs:",
    clean_df["product_id"].duplicated().sum()
)
print(
    "Duplicate SKUs:",
    clean_df["sku"].duplicated().sum()
)

Final shape: (500, 15)
Duplicate rows: 0
Duplicate product IDs: 0
Duplicate SKUs: 0


### Correcting the Data Type

In [77]:
# Convert product_id and review_count into integer columns
clean_df["product_id"] = pd.to_numeric(
    clean_df["product_id"],
    errors="coerce"
).astype("Int64")

clean_df["review_count"] = pd.to_numeric(
    clean_df["review_count"],
    errors="coerce"
).astype("Int64")


# Convert price-related columns into decimal/float columns
clean_df["price"] = pd.to_numeric(
    clean_df["price"],
    errors="coerce"
)

clean_df["original_price"] = pd.to_numeric(
    clean_df["original_price"],
    errors="coerce"
)


# Convert rating into a decimal/float column
clean_df["rating"] = pd.to_numeric(
    clean_df["rating"],
    errors="coerce"
)


# Convert in_stock into a Boolean column
clean_df["in_stock"] = clean_df["in_stock"].astype("boolean")


# Convert created_at into a proper date and time column
clean_df["created_at"] = pd.to_datetime(
    clean_df["created_at"],
    errors="coerce",
    utc=True
)

In [78]:
# Display column names, data types and non-null counts
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   product_id      500 non-null    Int64              
 1   product_name    500 non-null    string             
 2   vendor          500 non-null    string             
 3   category        500 non-null    string             
 4   price           500 non-null    float64            
 5   original_price  189 non-null    float64            
 6   rating          500 non-null    float64            
 7   review_count    500 non-null    Int64              
 8   in_stock        500 non-null    boolean            
 9   sku             500 non-null    string             
 10  tags            500 non-null    string             
 11  description     500 non-null    string             
 12  created_at      500 non-null    datetime64[ns, UTC]
 13  image_url       500 non-null    str

In [79]:
# Count missing values after converting the data types
clean_df[
    [
        "product_id",
        "price",
        "rating",
        "review_count",
        "in_stock",
        "created_at"
    ]
].isnull().sum()

product_id      0
price           0
rating          0
review_count    0
in_stock        0
created_at      0
dtype: int64

### Handle missing original prices and create discount columns

In [81]:
# Create a Boolean column that identifies discounted products
# True means an original price is available
# False means the product is not discounted
clean_df["is_discounted"] = clean_df["original_price"].notna()

In [82]:
# Calculate the difference between original price and current price
# Non-discounted products receive a discount amount of 0
clean_df["discount_amount"] = np.where(
    clean_df["is_discounted"],
    clean_df["original_price"] - clean_df["price"],
    0
)

# Round the discount amount to two decimal places
clean_df["discount_amount"] = (
    clean_df["discount_amount"].round(2)
)

In [83]:
# Calculate discount percentage only for discounted products
# Non-discounted products receive a discount percentage of 0
clean_df["discount_percentage"] = np.where(
    clean_df["is_discounted"],
    (
        (clean_df["original_price"] - clean_df["price"])
        / clean_df["original_price"]
    ) * 100,
    0
)

# Round the discount percentage to two decimal places
clean_df["discount_percentage"] = (
    clean_df["discount_percentage"].round(2)
)

In [84]:
# Display the original and newly created discount columns
clean_df[
    [
        "price",
        "original_price",
        "is_discounted",
        "discount_amount",
        "discount_percentage"
    ]
].head(10)

,price,original_price,is_discounted,discount_amount,discount_percentage
0,155.62,206.69,True,51.07,24.71
1,145.40,210.01,True,64.61,30.77
2,185.01,313.98,True,128.97,41.08
3,70.86,NaN,False,0.00,0.00
4,97.13,NaN,False,0.00,0.00
5,145.40,NaN,False,0.00,0.00
6,139.94,173.26,True,33.32,19.23
7,108.99,NaN,False,0.00,0.00
8,44.15,69.14,True,24.99,36.14
9,193.91,258.48,True,64.57,24.98


In [85]:
# Count discounted and non-discounted products
print(clean_df["is_discounted"].value_counts())

# Check the remaining missing original prices
print(
    "Missing original prices:",
    clean_df["original_price"].isnull().sum()
)

is_discounted
False    311
True     189
Name: count, dtype: int64
Missing original prices: 311


### Create a readable stock-status column

In [86]:
# Convert Boolean stock values into readable text
# True becomes "In Stock" and False becomes "Out of Stock"
clean_df["stock_status"] = np.where(
    clean_df["in_stock"] == True,
    "In Stock",
    "Out of Stock"
)

In [87]:
# Compare the original Boolean column with the new text column
clean_df[
    ["in_stock", "stock_status"]
].head(10)

,in_stock,stock_status
0,True,In Stock
1,False,Out of Stock
2,True,In Stock
3,False,Out of Stock
4,True,In Stock
5,True,In Stock
6,True,In Stock
7,True,In Stock
8,True,In Stock
9,True,In Stock


In [88]:
# Count in-stock and out-of-stock products
clean_df["stock_status"].value_counts()

stock_status
In Stock        427
Out of Stock     73
Name: count, dtype: int64

In [89]:
# Check for missing values in the new column
clean_df["stock_status"].isnull().sum()

0

### Create date-related columns

In [90]:
# Extract only the date portion from created_at
clean_df["created_date"] = clean_df["created_at"].dt.date


# Extract the year as an integer
clean_df["created_year"] = (
    clean_df["created_at"]
    .dt.year
    .astype("Int64")
)


# Extract the month number
# This will help sort month names correctly in Power BI
clean_df["created_month"] = (
    clean_df["created_at"]
    .dt.month
    .astype("Int64")
)


# Extract the complete month name
clean_df["created_month_name"] = (
    clean_df["created_at"]
    .dt.month_name()
)

In [91]:
# Display the original date and extracted date columns
clean_df[
    [
        "created_at",
        "created_date",
        "created_year",
        "created_month",
        "created_month_name"
    ]
].head(10)

,created_at,created_date,created_year,created_month,created_month_name
0,2022-08-29 00:00:00+00:00,2022-08-29,2022,8,August
1,2023-06-04 00:00:00+00:00,2023-06-04,2023,6,June
2,2023-04-17 00:00:00+00:00,2023-04-17,2023,4,April
3,2022-10-24 00:00:00+00:00,2022-10-24,2022,10,October
4,2022-08-07 00:00:00+00:00,2022-08-07,2022,8,August
5,2022-08-30 00:00:00+00:00,2022-08-30,2022,8,August
6,2023-08-19 00:00:00+00:00,2023-08-19,2023,8,August
7,2023-09-04 00:00:00+00:00,2023-09-04,2023,9,September
8,2023-10-16 00:00:00+00:00,2023-10-16,2023,10,October
9,2023-10-27 00:00:00+00:00,2023-10-27,2023,10,October


In [92]:
# Count products created in each year
clean_df["created_year"].value_counts().sort_index()

created_year
2022    238
2023    260
2024      2
Name: count, dtype: Int64

In [93]:
# Check whether any date conversion failed
clean_df[
    [
        "created_at",
        "created_date",
        "created_year",
        "created_month",
        "created_month_name"
    ]
].isnull().sum()

created_at            0
created_date          0
created_year          0
created_month         0
created_month_name    0
dtype: int64

### Create price categories

In [94]:
# Create price categories using fixed price ranges
# right=False means:
# 0–49.99 = Budget
# 50–99.99 = Affordable
# 100–149.99 = Premium
# 150 and above = Luxury

clean_df["price_category"] = pd.cut(
    clean_df["price"],
    bins=[0, 50, 100, 150, np.inf],
    labels=[
        "Budget",
        "Affordable",
        "Premium",
        "Luxury"
    ],
    right=False
)

In [95]:
# Display product prices with their assigned categories
clean_df[
    ["product_name", "price", "price_category"]
].head(10)

,product_name,price,price_category
0,Lightweight Probiotics,155.62,Luxury
1,Wireless LED Desk Lamp,145.40,Premium
2,Deluxe Tea Collection,185.01,Luxury
3,Essential Tire Gauge,70.86,Affordable
4,Smart Carving Tools,97.13,Affordable
5,Professional Dog Food,145.40,Premium
6,Smart Car Wash Kit,139.94,Premium
7,Premium Bluetooth Speaker,108.99,Premium
8,Vintage Tablet Stand,44.15,Budget
9,Portable Marker Pack,193.91,Luxury


In [96]:
# Count products belonging to each price category
clean_df["price_category"].value_counts(
    sort=False
)

price_category
Budget        112
Affordable    122
Premium       131
Luxury        135
Name: count, dtype: int64

In [97]:
# Check whether any products failed to receive a price category
clean_df["price_category"].isnull().sum()

0

### Create rating categories

In [98]:
# Create rating categories using defined rating ranges
# 1.0–3.4 = Average
# 3.5–3.9 = Good
# 4.0–4.4 = Very Good
# 4.5–5.0 = Excellent

clean_df["rating_category"] = pd.cut(
    clean_df["rating"],
    bins=[0, 3.5, 4.0, 4.5, 5.1],
    labels=[
        "Average",
        "Good",
        "Very Good",
        "Excellent"
    ],
    right=False
)

In [99]:
# Display ratings with their corresponding rating categories
clean_df[
    ["product_name", "rating", "rating_category"]
].head(10)

,product_name,rating,rating_category
0,Lightweight Probiotics,4.0,Very Good
1,Wireless LED Desk Lamp,4.6,Excellent
2,Deluxe Tea Collection,3.5,Good
3,Essential Tire Gauge,3.6,Good
4,Smart Carving Tools,3.5,Good
5,Professional Dog Food,4.6,Excellent
6,Smart Car Wash Kit,3.0,Average
7,Premium Bluetooth Speaker,4.5,Excellent
8,Vintage Tablet Stand,3.4,Average
9,Portable Marker Pack,4.2,Very Good


In [100]:
# Count products belonging to each rating category
clean_df["rating_category"].value_counts(
    sort=False
)

rating_category
Average      115
Good         117
Very Good    122
Excellent    146
Name: count, dtype: int64

In [101]:
# Check whether any ratings failed to receive a category
clean_df["rating_category"].isnull().sum()

0

### Validate the cleaned dataset

In [102]:
# Display the final number of rows and columns
print("Dataset shape:", clean_df.shape)


# Check duplicate records
print(
    "Duplicate rows:",
    clean_df.duplicated().sum()
)

print(
    "Duplicate product IDs:",
    clean_df["product_id"].duplicated().sum()
)

print(
    "Duplicate SKUs:",
    clean_df["sku"].duplicated().sum()
)

Dataset shape: (500, 25)
Duplicate rows: 0
Duplicate product IDs: 0
Duplicate SKUs: 0


In [103]:
# Count products containing a missing or non-positive current price
invalid_prices = (
    clean_df["price"].isnull()
    | (clean_df["price"] <= 0)
).sum()


# Count ratings outside the valid range of 1 to 5
invalid_ratings = (
    clean_df["rating"].isnull()
    | ~clean_df["rating"].between(1, 5)
).sum()


# Count products containing negative or missing review counts
invalid_reviews = (
    clean_df["review_count"].isnull()
    | (clean_df["review_count"] < 0)
).sum()


# Count discounted products where original price is not greater than current price
invalid_discount_prices = (
    clean_df["is_discounted"]
    & (
        clean_df["original_price"]
        <= clean_df["price"]
    )
).sum()


# Display the validation results
print("Invalid prices:", invalid_prices)
print("Invalid ratings:", invalid_ratings)
print("Invalid review counts:", invalid_reviews)
print(
    "Invalid discounted prices:",
    invalid_discount_prices
)

Invalid prices: 0
Invalid ratings: 0
Invalid review counts: 0
Invalid discounted prices: 0


In [104]:
# Display the missing-value count for every column
clean_df.isnull().sum()

product_id               0
product_name             0
vendor                   0
category                 0
price                    0
original_price         311
rating                   0
review_count             0
in_stock                 0
sku                      0
tags                     0
description              0
created_at               0
image_url                0
product_url              0
is_discounted            0
discount_amount          0
discount_percentage      0
stock_status             0
created_date             0
created_year             0
created_month            0
created_month_name       0
price_category           0
rating_category          0
dtype: int64

In [105]:
# Display a sample of the important cleaned columns
clean_df[
    [
        "product_id",
        "product_name",
        "vendor",
        "category",
        "price",
        "original_price",
        "is_discounted",
        "discount_amount",
        "discount_percentage",
        "rating",
        "rating_category",
        "stock_status",
        "price_category"
    ]
].head(10)

,product_id,product_name,vendor,category,price,original_price,is_discounted,discount_amount,discount_percentage,rating,rating_category,stock_status,price_category
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,True,51.07,24.71,4.0,Very Good,In Stock,Luxury
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,True,64.61,30.77,4.6,Excellent,Out of Stock,Premium
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,True,128.97,41.08,3.5,Good,In Stock,Luxury
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,False,0.00,0.00,3.6,Good,Out of Stock,Affordable
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,False,0.00,0.00,3.5,Good,In Stock,Affordable
5,6,Professional Dog Food,StyleCraft,Pet Supplies,145.40,NaN,False,0.00,0.00,4.6,Excellent,In Stock,Premium
6,7,Smart Car Wash Kit,UrbanThread,Automotive,139.94,173.26,True,33.32,19.23,3.0,Average,In Stock,Premium
7,8,Premium Bluetooth Speaker,AutoParts Pro,Electronics,108.99,NaN,False,0.00,0.00,4.5,Excellent,In Stock,Premium
8,9,Vintage Tablet Stand,GlowUp,Electronics,44.15,69.14,True,24.99,36.14,3.4,Average,In Stock,Budget
9,10,Portable Marker Pack,VitalLife,Art,193.91,258.48,True,64.57,24.98,4.2,Very Good,In Stock,Luxury


### Save the cleaned dataset

In [106]:
# Save the cleaned DataFrame as a new CSV file
# index=False prevents Pandas from adding an unnecessary index column
clean_df.to_csv(
    "products_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [107]:
# Read the saved CSV into a temporary DataFrame
check_df = pd.read_csv(
    "products_cleaned.csv"
)

# Display its dimensions
print("Saved dataset shape:", check_df.shape)

# Display the first five records
check_df.head()

Saved dataset shape: (500, 25)


,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,...,is_discounted,discount_amount,discount_percentage,stock_status,created_date,created_year,created_month,created_month_name,price_category,rating_category
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4.0,52,True,SKU-HEA-0001,...,True,51.07,24.71,In Stock,2022-08-29,2022,8,August,Luxury,Very Good
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,4.6,12,False,SKU-ELE-0002,...,True,64.61,30.77,Out of Stock,2023-06-04,2023,6,June,Premium,Excellent
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,3.5,211,True,SKU-FOO-0003,...,True,128.97,41.08,In Stock,2023-04-17,2023,4,April,Luxury,Good
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,3.6,47,False,SKU-AUT-0004,...,False,0.00,0.00,Out of Stock,2022-10-24,2022,10,October,Affordable,Good
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,3.5,353,True,SKU-ART-0005,...,False,0.00,0.00,In Stock,2022-08-07,2022,8,August,Affordable,Good


In [108]:
# Display all column names from the saved dataset
check_df.columns.tolist()

['product_id',
 'product_name',
 'vendor',
 'category',
 'price',
 'original_price',
 'rating',
 'review_count',
 'in_stock',
 'sku',
 'tags',
 'description',
 'created_at',
 'image_url',
 'product_url',
 'is_discounted',
 'discount_amount',
 'discount_percentage',
 'stock_status',
 'created_date',
 'created_year',
 'created_month',
 'created_month_name',
 'price_category',
 'rating_category']

### Prepare the dataset for MySQL

In [109]:
# Create a separate copy for MySQL
# This keeps clean_df unchanged
sql_df = clean_df.copy()

In [110]:
# Convert True/False into 1/0 for easier MySQL import
sql_df["in_stock"] = sql_df["in_stock"].astype(int)
sql_df["is_discounted"] = sql_df["is_discounted"].astype(int)

In [111]:
# Convert created_at into a MySQL-compatible date-time format
sql_df["created_at"] = pd.to_datetime(
    sql_df["created_at"],
    errors="coerce",
    utc=True
).dt.strftime("%Y-%m-%d %H:%M:%S")


# Convert created_date into a MySQL-compatible date format
sql_df["created_date"] = pd.to_datetime(
    sql_df["created_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

In [112]:
# Convert Pandas category columns into regular text columns
sql_df["price_category"] = (
    sql_df["price_category"].astype("string")
)

sql_df["rating_category"] = (
    sql_df["rating_category"].astype("string")
)

In [113]:
# Save missing original prices as blank values
# MySQL can import these blank values as NULL
sql_df.to_csv(
    "products_sql.csv",
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

print("MySQL-ready dataset saved successfully!")

MySQL-ready dataset saved successfully!


In [114]:
# Display the exact location and shape of the MySQL-ready file
print(os.path.abspath("products_sql.csv"))
print("Dataset shape:", sql_df.shape)

sql_df.head()

C:\Users\Khushi\Downloads\products_sql.csv
Dataset shape: (500, 25)


,product_id,product_name,vendor,category,price,original_price,rating,review_count,in_stock,sku,...,is_discounted,discount_amount,discount_percentage,stock_status,created_date,created_year,created_month,created_month_name,price_category,rating_category
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,4.0,52,1,SKU-HEA-0001,...,1,51.07,24.71,In Stock,2022-08-29,2022,8,August,Luxury,Very Good
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,4.6,12,0,SKU-ELE-0002,...,1,64.61,30.77,Out of Stock,2023-06-04,2023,6,June,Premium,Excellent
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,3.5,211,1,SKU-FOO-0003,...,1,128.97,41.08,In Stock,2023-04-17,2023,4,April,Luxury,Good
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,3.6,47,0,SKU-AUT-0004,...,0,0.00,0.00,Out of Stock,2022-10-24,2022,10,October,Affordable,Good
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,3.5,353,1,SKU-ART-0005,...,0,0.00,0.00,In Stock,2022-08-07,2022,8,August,Affordable,Good
